# Chapter 7 — Working with Text Data

Chapter ini membahas bagaimana data berupa teks dapat diubah menjadi representasi numerik sehingga dapat digunakan oleh algoritma Machine Learning.

Topik utama:
1. Representasi teks
2. Bag-of-Words
3. CountVectorizer
4. Stopwords
5. TF-IDF
6. Word n-grams dan Character n-grams
7. Text Classification
8. Pipeline untuk text processing
9. Hyperparameter tuning
10. Interpretasi model teks
11. Topic Modeling dengan Latent Dirichlet Allocation (LDA)

## 1. Mengapa Text Data Perlu Diproses?

Algoritma Machine Learning umumnya bekerja dengan data numerik.

Sementara itu, data teks berbentuk kata atau kalimat, misalnya:

- "I love this movie"
- "The movie was terrible"

Oleh karena itu, teks perlu diubah menjadi representasi numerik.

Salah satu pendekatan paling umum adalah **Bag-of-Words**, yaitu merepresentasikan dokumen berdasarkan kata-kata yang muncul di dalamnya.

Representasi teks biasanya menghasilkan matriks dengan jumlah fitur yang sangat banyak dan sebagian besar nilainya bernilai 0. Oleh karena itu, scikit-learn menggunakan **sparse matrix** agar penyimpanan lebih efisien.

In [ ]:
texts = [
    "I love this movie it is amazing",
    "This movie is wonderful and great",
    "I really enjoyed this film",
    "The story was beautiful and inspiring",
    "The acting was excellent",
    "This is a fantastic movie",
    "I enjoyed the story very much",
    "The movie was really good",

    "I hate this movie it was terrible",
    "This movie is boring and bad",
    "I really disliked this film",
    "The story was boring and disappointing",
    "The acting was awful",
    "This is a terrible movie",
    "I did not enjoy the story",
    "The movie was really bad"
]

y = [
    1, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 0, 0
]

print("Jumlah dokumen:", len(texts))
print("Contoh dokumen:")
print(texts[0])

### Label

Pada dataset sederhana ini:

- `1` = review positif
- `0` = review negatif

Dataset ini hanya digunakan untuk memahami konsep text classification.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Jumlah data training:", len(X_train))
print("Jumlah data testing:", len(X_test))

## 4. Bag-of-Words

Bag-of-Words (BoW) merupakan cara sederhana untuk merepresentasikan teks.

Langkahnya:

1. Mengumpulkan seluruh kata dari dokumen.
2. Setiap kata menjadi sebuah fitur.
3. Menghitung berapa kali kata tersebut muncul dalam setiap dokumen.

Contoh:

Dokumen:

- "I like coffee"
- "I like tea"

Vocabulary:

`I`, `like`, `coffee`, `tea`

Dokumen kemudian direpresentasikan sebagai angka berdasarkan jumlah kemunculan setiap kata.

Kelemahan Bag-of-Words:
- Tidak memperhatikan urutan kata.
- Vocabulary dapat menjadi sangat besar.
- Banyak nilai yang bernilai 0.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

print("Ukuran matriks training:", X_train_counts.shape)
print("Ukuran matriks testing:", X_test_counts.shape)

print("\nBeberapa vocabulary:")
print(vectorizer.get_feature_names_out()[:15])

### `fit_transform()` dan `transform()`

Pada data training:

```python
vectorizer.fit_transform(X_train)


---

# 🟦 Cell 9 — Markdown
## 6. Stopwords

```markdown
## 6. Stopwords

Stopwords adalah kata-kata yang sering muncul tetapi terkadang kurang memberikan informasi penting untuk suatu tugas.

Contohnya dalam bahasa Inggris:

- the
- is
- a
- and
- this

`CountVectorizer` dapat menghapus stopwords menggunakan:

```python
stop_words="english"


---

# 🟦 Cell 10 — Code

```python
vectorizer_stop = CountVectorizer(stop_words="english")

X_train_stop = vectorizer_stop.fit_transform(X_train)

print("Ukuran tanpa stopwords:", X_train_counts.shape)
print("Ukuran dengan stopwords:", X_train_stop.shape)

print("\nVocabulary:")
print(vectorizer_stop.get_feature_names_out())

## 7. Word n-Grams

Bag-of-Words biasanya hanya melihat satu kata pada satu waktu atau **unigram**.

Namun, kita juga dapat mempertimbangkan kombinasi beberapa kata.

Contoh:

Kalimat:

"I really love this movie"

### Unigram
- I
- really
- love
- this
- movie

### Bigram
- I really
- really love
- love this
- this movie

Parameter:

```python
ngram_range=(1, 2)


---

# 🟦 Cell 12 — Code

```python
bigram_vectorizer = CountVectorizer(
    ngram_range=(1, 2)
)

X_bigram = bigram_vectorizer.fit_transform(X_train)

print("Ukuran matriks:", X_bigram.shape)

print("\nContoh fitur:")
print(bigram_vectorizer.get_feature_names_out()[:30])

Selain berdasarkan kata, teks juga dapat direpresentasikan berdasarkan karakter.

Contohnya:

```python
analyzer="char"


---

# 🟦 Cell 14 — Code

```python
char_vectorizer = CountVectorizer(
    analyzer="char",
    ngram_range=(3, 5)
)

X_char = char_vectorizer.fit_transform(X_train)

print("Ukuran matriks character n-gram:", X_char.shape)

## 9. TF-IDF

Bag-of-Words hanya menghitung jumlah kemunculan kata.

TF-IDF memberikan bobot yang lebih tinggi pada kata yang penting bagi suatu dokumen dan menurunkan bobot kata yang muncul di banyak dokumen.

TF-IDF terdiri dari:

- **TF (Term Frequency)** → seberapa sering kata muncul dalam dokumen.
- **IDF (Inverse Document Frequency)** → seberapa jarang kata tersebut muncul di seluruh dokumen.

Secara sederhana:

TF-IDF = TF × IDF

Kata yang muncul di hampir semua dokumen cenderung mendapatkan bobot lebih rendah.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Ukuran training:", X_train_tfidf.shape)
print("Ukuran testing:", X_test_tfidf.shape)

print("\nContoh vocabulary:")
print(tfidf.get_feature_names_out()[:15])

## 10. Text Classification dengan Naive Bayes

Setelah teks diubah menjadi angka, representasi tersebut dapat digunakan oleh algoritma Machine Learning.

Salah satu algoritma yang sering digunakan untuk text classification adalah **Multinomial Naive Bayes**.

Pada contoh ini model akan memprediksi apakah sebuah review termasuk:

- positif
- negatif

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

model_nb = MultinomialNB()

model_nb.fit(X_train_tfidf, y_train)

y_pred_nb = model_nb.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

## 11. Text Classification dengan Logistic Regression

Logistic Regression juga dapat digunakan untuk melakukan klasifikasi teks.

Model akan mempelajari hubungan antara fitur kata dan kelas target.

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=2000)

model_lr.fit(X_train_tfidf, y_train)

y_pred_lr = model_lr.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

## 12. Pipeline

Pipeline memungkinkan proses preprocessing dan machine learning digabungkan menjadi satu alur.

Contohnya:

Text
↓
TF-IDF
↓
Logistic Regression
↓
Prediction

Keuntungan Pipeline:
- kode lebih rapi
- preprocessing dan model selalu digunakan bersama
- mengurangi risiko data leakage
- lebih mudah digunakan bersama cross-validation dan GridSearchCV

In [ ]:
from sklearn.pipeline import Pipeline

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LogisticRegression(max_iter=2000))
])

text_pipeline.fit(X_train, y_train)

y_pred = text_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

## 13. Hyperparameter Tuning

Pada text classification terdapat beberapa parameter yang dapat diuji, misalnya:

- `ngram_range`
- `min_df`
- `max_df`
- `C` pada Logistic Regression

Kita dapat menggunakan GridSearchCV untuk mencari kombinasi parameter yang memberikan hasil cross-validation terbaik.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "model__C": [0.1, 1, 10]
}

grid = GridSearchCV(
    text_pipeline,
    param_grid,
    cv=3
)

grid.fit(X_train, y_train)

print("Parameter terbaik:")
print(grid.best_params_)

print("\nScore cross-validation terbaik:")
print(grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

y_pred_grid = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_grid))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_grid))

## 14. Investigating Model Coefficients

Salah satu keuntungan menggunakan model linear seperti Logistic Regression adalah kita dapat melihat fitur yang memiliki hubungan dengan prediksi model.

Koefisien positif menunjukkan hubungan dengan kelas positif, sedangkan koefisien negatif menunjukkan hubungan dengan kelas negatif.

Interpretasi ini bergantung pada model dan representasi fitur yang digunakan.

In [ ]:
import numpy as np

tfidf_model = best_model.named_steps["tfidf"]
lr_model = best_model.named_steps["model"]

feature_names = tfidf_model.get_feature_names_out()
coefficients = lr_model.coef_[0]

# 10 fitur dengan koefisien paling positif
positive_idx = np.argsort(coefficients)[-10:][::-1]

print("Fitur yang paling berhubungan dengan kelas positif:")
for idx in positive_idx:
    print(feature_names[idx], ":", round(coefficients[idx], 3))

In [ ]:
negative_idx = np.argsort(coefficients)[:10]

print("Fitur yang paling berhubungan dengan kelas negatif:")
for idx in negative_idx:
    print(feature_names[idx], ":", round(coefficients[idx], 3))

## 15. Topic Modeling

Selain supervised learning, text data juga dapat dianalisis menggunakan unsupervised learning.

Salah satu tugasnya adalah **topic modeling**.

Tujuannya adalah menemukan topik yang muncul dalam kumpulan dokumen tanpa menyediakan label secara manual.

Salah satu metode yang digunakan adalah:

**Latent Dirichlet Allocation (LDA)**

LDA mencoba menemukan kelompok kata yang sering muncul bersama dan menggunakannya untuk membentuk topik.

Penting:
Topik yang dihasilkan merupakan hasil model dan tidak selalu memiliki interpretasi yang benar secara otomatis. Kita perlu melihat kata-kata utama untuk memberikan interpretasi.

In [ ]:
topic_docs = [
    "football team player match goal stadium",
    "football player scored goal during the match",
    "team won the football championship",
    "basketball player scored points in the game",

    "computer software technology artificial intelligence",
    "machine learning model uses computer data",
    "technology company develops new software",
    "artificial intelligence improves computer systems",

    "restaurant food delicious cooking kitchen",
    "chef prepares delicious food in the restaurant",
    "cooking recipes use fresh ingredients",
    "restaurant serves healthy and tasty meals"
]

print("Jumlah dokumen:", len(topic_docs))

In [ ]:
topic_vectorizer = CountVectorizer(
    stop_words="english"
)

X_topics = topic_vectorizer.fit_transform(topic_docs)

print("Ukuran matriks:", X_topics.shape)

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(
    n_components=3,
    random_state=42
)

lda.fit(X_topics)

print("LDA berhasil dilatih.")

In [ ]:
terms = topic_vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[-8:][::-1]

    print(f"\nTopic {topic_idx + 1}:")
    print(", ".join(terms[i] for i in top_indices))

### Interpretasi

Setelah menjalankan LDA, kita dapat melihat kata-kata utama pada setiap topic.

Contohnya, model mungkin menemukan kelompok kata seperti:

- football, player, team, goal → topik olahraga
- computer, software, technology, AI → topik teknologi
- restaurant, food, cooking, chef → topik makanan

Interpretasi tersebut dilakukan berdasarkan kata-kata dengan bobot tertinggi pada masing-masing topic.

## 16. Perbandingan Representasi Teks

| Metode | Fungsi |
|---|---|
| Bag-of-Words | Menghitung kemunculan kata |
| CountVectorizer | Mengubah teks menjadi matriks jumlah kata |
| Stopwords | Mengurangi kata yang dianggap kurang informatif |
| Word n-grams | Mempertimbangkan kombinasi beberapa kata |
| Character n-grams | Menggunakan kombinasi karakter |
| TF-IDF | Memberikan bobot berdasarkan pentingnya kata |
| LDA | Menemukan topic secara unsupervised |

Pemilihan representasi bergantung pada karakteristik data dan tujuan machine learning.

## 17. Kesimpulan

Pada chapter ini telah dipelajari bagaimana data teks dapat digunakan dalam Machine Learning.

Hal-hal utama yang dipelajari:

1. Text data perlu diubah menjadi representasi numerik.
2. Bag-of-Words merupakan pendekatan sederhana untuk merepresentasikan teks.
3. CountVectorizer menghitung kemunculan kata dalam dokumen.
4. Stopwords dapat digunakan untuk mengurangi kata yang kurang informatif.
5. N-grams memungkinkan model mempertimbangkan kombinasi kata atau karakter.
6. TF-IDF memberikan bobot berdasarkan tingkat kepentingan kata.
7. Representasi teks dapat digunakan untuk classification.
8. Pipeline menggabungkan preprocessing dan model dalam satu alur.
9. GridSearchCV dapat digunakan untuk mencari hyperparameter terbaik.
10. Koefisien model linear dapat digunakan untuk melihat fitur yang berpengaruh.
11. LDA dapat digunakan untuk melakukan topic modeling secara unsupervised.